In [ ]:
import json

# Load configuration from config.json
with open('config.json', 'r') as f:
    config = json.load(f)

print("Configuration loaded:")
print(json.dumps(config, indent=4))

# Configuration

Before running the code, edit the `config.json` file in the project root to specify the following attributes:

- `lat`: Latitude of the solar panel location (e.g., 51.2 for London)
- `lon`: Longitude of the solar panel location (e.g., -0.1)
- `kwp`: System peak power in kilowatts-peak (e.g., 10.0)
- `tilt`: Panel tilt angle in degrees (e.g., 35)
- `azimuth`: Panel orientation in degrees (0=South, -90=East, 90=West)
- `yield_factor`: Efficiency factor accounting for losses (e.g., 0.80)

The notebook will load these values automatically. The API forecasts use 15-minute intervals for improved granularity.

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

def calculate_solar_forecast(lat, lon, kwp, tilt, azimuth, yield_factor=0.75):
    # 1. Create a 15-minute time range for the next 24 hours
    start_time = datetime.now().replace(minute=0, second=0, microsecond=0)
    times = [start_time + timedelta(minutes=15*i) for i in range(96)]
    
    # 2. Basic Constants
    solar_constant = 1000  # Standard Test Condition (W/m2)
    day_of_year = datetime.now().timetuple().tm_yday
    
    # 3. Calculate Solar Position (Simplified)
    # Declination angle (approximate for the day)
    declination = 23.45 * np.sin(np.radians(360/365 * (day_of_year - 81)))
    
    results = []
    for t in times:
        # Hour angle: 15 degrees per hour from solar noon (12:00)
        hour_angle = 15 * (t.hour + t.minute/60 - 12)
        
        # Solar Zenith Angle (angle from directly overhead)
        cos_zenith = (np.sin(np.radians(lat)) * np.sin(np.radians(declination)) + 
                      np.cos(np.radians(lat)) * np.cos(np.radians(declination)) * 
                      np.cos(np.radians(hour_angle)))
        zenith = np.degrees(np.arccos(np.clip(cos_zenith, -1, 1)))
        
        # If sun is below horizon, output is 0
        if zenith > 90:
            power_output = 0
        else:
            # 4. Calculate Incident Angle on Tilted Surface
            # 0 azimuth = South, 90 = West, -90 = East
            # This formula finds how 'directly' the sun hits the panel
            cos_incidence = (np.cos(np.radians(zenith)) * np.cos(np.radians(tilt)) + 
                             np.sin(np.radians(zenith)) * np.sin(np.radians(tilt)) * 
                             np.cos(np.radians(hour_angle - azimuth)))
            
            # 5. Final Power Calculation (kW)
            # Power = Peak Power * Efficiency * (Actual Irradiance / 1000W/m2)
            # We assume a clear sky irradiance of ~1000W/m2 * cos(zenith)
            irradiance = solar_constant * max(0, cos_incidence)
            power_output = kwp * yield_factor * (irradiance / 1000)
            
        results.append({"Time": t.strftime("%H:%M"), "Power_kW": round(power_output, 3)})

    return pd.DataFrame(results)

# Use the loaded config
forecast_df = calculate_solar_forecast(**config)
print(forecast_df.head(20)) # Display first 5 hours

In [ ]:
import requests
import pandas as pd

def get_solar_weather(lat, lon, kwp, tilt_angle):
    base_url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "minutely_15": "direct_radiation,diffuse_radiation,temperature_2m",
        "forecast_days": 1,
        "timezone": "auto",
    }

    response = requests.get(base_url, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if "minutely_15" not in payload:
        raise ValueError("Open-Meteo response has no 'minutely_15' data")
    data = payload["minutely_15"]

    df = pd.DataFrame(data)
    df['time'] = pd.to_datetime(df['time'])

    # Constants
    system_loss = 0.85  # 15% loss for dust, inverter, wires
    temp_coeff = -0.004  # Panels lose 0.4% efficiency per degree above 25C

    # Simplified projection: Adjusting direct radiation by tilt
    # A rough estimate: radiation * cos(tilt_difference)
    # For a perfect calculation, you'd use the sun's position vs panel angle
    tilt_correction = 0.9  # Assuming panel is well-aligned

    def calculate_output(row):
        # Calculate effective irradiance
        effective_irradiance = (row['direct_radiation'] * tilt_correction) + row['diffuse_radiation']

        # Base Power: (Irradiance / 1000W/m2) * kWp
        power = (effective_irradiance / 1000) * kwp * system_loss

        # Temperature Correction
        if row['temperature_2m'] > 25:
            power *= (1 + (row['temperature_2m'] - 25) * temp_coeff)

        return max(0, power)

    df['predicted_kw'] = df.apply(calculate_output, axis=1)
    return df[['time', 'predicted_kw', 'temperature_2m']]

# Use the loaded config
forecast = get_solar_weather(config['lat'], config['lon'], config['kwp'], config['tilt'])
print(forecast.to_string())

# Doublecheck with real data

In [ ]:
from pathlib import Path
import re
import pandas as pd

pattern = re.compile(
    r"^\s*\d+\s+(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})\s+([-+]?\d*\.?\d+)\s+([-+]?\d*\.?\d+)\s*$"
    )

def load_solar_txt(file_path: Path) -> pd.DataFrame:
    rows = []
    for line in file_path.read_text(encoding="utf-8").splitlines():
        match = pattern.match(line)
        if match:
            rows.append(match.groups())

    df = pd.DataFrame(rows, columns=["time", "predicted_kw", "temperature_2m"])
    df["time"] = pd.to_datetime(df["time"])
    for col in ["predicted_kw", "temperature_2m"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["source"] = file_path.stem.split()[0]
    return df

txt_files = sorted(Path("data/validate").glob("*.txt"))
df_all = pd.concat([load_solar_txt(fp) for fp in txt_files], ignore_index=True)

df_all.head()

,time,predicted_kw,temperature_2m,source
0,2026-03-09 00:00:00,0.0,9.6,sun1_kw
1,2026-03-09 00:15:00,0.0,9.3,sun1_kw
2,2026-03-09 00:30:00,0.0,9.0,sun1_kw
3,2026-03-09 00:45:00,0.0,8.7,sun1_kw
4,2026-03-09 01:00:00,0.0,8.5,sun1_kw


In [ ]:
# 15-minute values: kWh per row = kW * 0.25 hours
df_all["energy_kwh"] = df_all["predicted_kw"] * 0.25

energy_by_source = df_all.groupby("source", as_index=False)["energy_kwh"].sum()
total_kwh = energy_by_source["energy_kwh"].sum()

print(energy_by_source)
print(f"Total energy (all txt files): {total_kwh:.3f} kWh")

    source  energy_kwh
0  sun1_kw   16.281972
1  sun2_kw    5.913287
Total energy (all txt files): 22.195 kWh
